# 第三届未央城赛道一的简单demo！
欢迎大家参加第三届未央城赛事🥳，这是一个简单的入门教程   
在这个教程中我会带着你用langchain和ReAct框架搭建一个简单的数理智能体，仅作为抛砖引玉，期待大家的发挥！   

## 一、准备事项
### 一、大模型的准备
让我们回归最基本的问题：   
1.LangChain 是什么？   
简单来说，LangChain 是一个用来“粘合”大模型（LLM）和各种外部工具的开发框架（SDK）。   
如果没有 LangChain，直接调用大模型（比如 GPT-4）就像是你在和一个“只有大脑，没有手脚”的天才对话。他很聪明，但他记不住你们之前的聊天（没有内存），他也不能上网、不能翻看你电脑里的 Excel 表格（没有工具）。   
**LangChain 的作用就是给这个大脑装上“手脚”和“外挂”：**   
Chains（链）：把动作串联起来。比如：先读取用户的问题 -> 翻译成英文 -> 发给大模型 -> 把答案翻译回中文。   
Agents（智能体）：让大模型自己决定该干什么。比如用户问“进动和章动分别是什么？”，LangChain 会让大模型思考，然后决定去调用“物理知识接口”，而不是直接瞎编。   
Retrieval（检索/RAG）：给大模型外挂知识库。比如让它读取shuo的基物1教材，然后回答相关问题。   
Memory（记忆）：让大模型记住上下文，能进行多轮对话。   
### 一句话总结：LangChain 是大模型应用开发的脚手架，它帮你把繁琐的代码封装好了，让你能快速搭出一个 AI 应用。   
2.ReAct范式是什么？   
ReAct 是 Reasoning（推理） + Acting（行动） 的缩写。它的流程是：   
Thinking（思考）：大模型看用户的问题，心想：“用户问的是数理问题，我应该先去查公式。”   
Action（行动）：大模型输出指令，调用你写的“数理知识库工具”。   
Observation（观察）：工具返回了结果（比如 F=ma）。   
Thinking（再思考）：大模型心想：“公式有了，现在我需要计算具体的数值。”   
Action（再行动）：大模型调用“计算器工具”。   
Final Answer（最终回答）：大模型输出最终结果。   
**从这里我们可以得出结论：我们首先需要一个大模型的API KEY**   

一个方便获取的地方： https://easycompute.cs.tsinghua.edu.cn/login   
在这里申请获取大模型KEY后就可以使用了，例如：   

In [ ]:
from langchain_openai import ChatOpenAI
from pathlib import Path

api_key_file = Path("api_key.txt")
if not api_key_file.exists():
    raise FileNotFoundError("未找到 api_key.txt，请在项目根目录创建该文件并写入你的 API Key。")

API_KEY = api_key_file.read_text(encoding="utf-8").strip()
if not API_KEY:
    raise ValueError("api_key.txt 为空，请写入有效的 API Key。")

llm = ChatOpenAI(
    model="DeepSeek-V3.2",
    api_key=API_KEY,
    base_url="https://llmapi.paratera.com",
    temperature=0,
    streaming=True
)

当然，大模型调试也有讲究，建议选思考时间不太长的大模型来调试，如果选deepseek-R1这种的思考类大模型你有福享了……   
不过放心！最后的大模型是统一的，我们不会在这里创造不公平因素！   
### 2.langchain的配置   
这个地方参加过第二届未央城的同学肯定很熟悉，首先我们要创造虚拟环境，其次将相关的模块导入来用   
创建虚拟环境（以**cmd**中为例，分别为切换到D盘，切换到指定文件夹，创建虚拟环境，激活（进入）虚拟环境，安装相应依赖）   
指定版本号是为了让这些模块配套，防止你的程序之后找不到相关模块中的函数   

In [ ]:
D:
cd weyoungcity_3
python -m venv venv
.\venv\Scripts\activate
pip install langchain==0.3.25 langchain-core==0.3.65 langchain-community==0.3.24 langchain-openai==0.3.16
pip install numexpr sentence-transformers faiss-cpu

这之后要告诉编辑器有这个东西（以VScode为例）   
1.按 Ctrl + Shift + P。   
2.输入 Python: Select Interpreter。   
3.选择列表里带有 venv 字样的那个（或者选 Enter interpreter path -> 找到你项目文件夹下的 venv/Scripts/python.exe）。   
## 二、搭建智能体的可用工具列表：一个数理知识库和一个数学计算工具   
数理知识库提供模型可以读取的信息，数学计算工具就顾名思义了，程序来算肯定比大模型自己来要快   

In [ ]:
# 这是数理知识库的构建过程
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings  # 根据不同API更换
from langchain.tools import Tool

# 第1步: 直接加载原始 PDF
# 使用 PyPDFLoader 读取 PDF 文件。这里假设 PDF 文件名为 "数理公式 (全).pdf"
loader = PyPDFLoader("数理公式 (全).pdf")

# 使用 loader.load() 方法加载 PDF 文件中的所有页面，并将其存储在 raw_pages 变量中
raw_pages = loader.load()

# 第2步: 直接按字符数切分
# 创建一个文本分割器，名为 RecursiveCharacterTextSplitter，其目的是分割文档为更小的部分
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # 每个切分的片段最大为500个字符
    chunk_overlap=50 # 片段之间有50个字符的重叠以防止公式被切断
)

# 使用创建的文本分割器将加载的 PDF 文件内容分割成多个 smaller documents
docs = text_splitter.split_documents(raw_pages)

# 第3步: 建立搜索索引
# 创建一个 FAISS 向量存储。在计算文档向量时使用 OpenAIEmbeddings，以便进行快速的相似性搜索
vectorstore = FAISS.from_documents(docs, OpenAIEmbeddings())

# 第4步: 定义直接检索工具
# 创建一个搜索函数，用于执行快速搜索任务
def quick_search(query: str):
    # 使用 FAISS 向量存储执行相似性搜索，返回与查询最相似的3个文档片段
    search_results = vectorstore.similarity_search(query, k=3)
    # 将搜索结果汇总为一个字符串，结果片段之间用 "---" 分隔
    return "\n---\n".join([res.page_content for res in search_results])

# 定义一个工具来进行快速参考
reference_tool = Tool(
    name="Quick_Reference",
    func=quick_search,
    description="直接搜索 PDF 文档中的原始文本。当你需要寻找特定的物理公式或数学定义时使用。"
)

# 现在，可以使用 reference_tool 执行快速搜索任务，以查找特定的物理公式或数学定义。

In [ ]:
# 这是数学计算工具的搭建过程
import sys
from io import StringIO
from langchain.tools import tool

@tool
def physics_math_solver(query: str):
    """
    一个数理逻辑计算引擎。
    输入：一段符合 Python 语法的数学计算代码。
    功能：利用 sympy 处理符号运算（求导、积分、方程求解）或利用 math 处理数值计算。
    """
    # 第1步: 准备一个“沙盒”环境，防止直接操作主程序
    # 创建一个安全的全局环境字典，包含常用的数学库，避免用户需要重复导入
    safe_globals = {
        "sympy": __import__("sympy"),  # 导入 sympy 进行符号数学运算
        "math": __import__("math"),    # 导入 math 处理基本数学运算
        "np": __import__("numpy")      # 导入 numpy 进行更多的数值计算
    }
    
    # 第2步: 捕获代码运行的输出结果
    # 备份当前的标准输出位置，以便恢复
    old_stdout = sys.stdout
    # 重定向标准输出到一个字符串缓冲区
    redirected_output = sys.stdout = StringIO()
    
    try:
        # 使用 exec() 函数在指定的安全全局环境中执行用户提供的代码
        exec(query, safe_globals)
        # 代码执行完成后，恢复标准输出
        sys.stdout = old_stdout
        # 返回执行代码后的输出结果
        return redirected_output.getvalue()
    except Exception as e:
        # 若代码执行过程中发生异常，恢复标准输出
        sys.stdout = old_stdout
        # 返回错误信息
        return f"代码执行出错: {str(e)}"

## 三、检索式问答链和ReAct范式智能体的创建   
这个的基本概念已经讨论过了，下面给出可运行的代码   
共分为两段，第一段是我用的一个txt形式的知识库创建的代码，这也可以看出知识库构建形式有很多，第二段是整体可运行程序（记得先运行第一段哦）   
知识库可以从这里查看：https://cloud.tsinghua.edu.cn/d/016f473aafe444d1aae5/
什么？你问我为什么写这么长？还不是因为我让deepseek输出“final answer”它老输出“最终答案”导致必须加特判嘛

In [ ]:
import os
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

file_path = "knowledge.txt"
db_save_path = "faiss_index"

print(f"正在读取文件: {file_path}...")

loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()

text_splitter = CharacterTextSplitter(separator="\n", chunk_size=200, chunk_overlap=40)
docs = text_splitter.split_documents(documents)

print(f"数据加载完成，共切分为 {len(docs)} 条知识片段。")
print("正在向量化并构建索引...")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local(db_save_path)

print(f"✅ 知识库已保存到: {db_save_path}")

In [ ]:
import os
import re
import sys
from io import StringIO

from langchain.agents import create_react_agent, AgentExecutor
from langchain.agents.output_parsers import ReActSingleInputOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.agents import AgentFinish
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool

# =======================================================
# 第一步：配置大模型
# =======================================================
api_key_file = "api_key.txt"
if not os.path.exists(api_key_file):
    raise FileNotFoundError("未找到 api_key.txt，请在项目根目录创建该文件并写入你的 API Key。")

with open(api_key_file, "r", encoding="utf-8") as f:
    API_KEY = f.read().strip()

if not API_KEY:
    raise ValueError("api_key.txt 为空，请写入有效的 API Key。")

llm = ChatOpenAI(
    model="DeepSeek-V3.2",
    api_key=API_KEY,
    base_url="https://llmapi.paratera.com",
    temperature=0,
    streaming=True
)

# =======================================================
# 第二步：加载知识库（FAISS + PDF）
# =======================================================

# ---------- FAISS 知识库 ----------
print("正在加载本地知识库...")
db_load_path = "faiss_index"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

try:
    vectorstore = FAISS.load_local(db_load_path, embeddings, allow_dangerous_deserialization=True)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
    print("✅ FAISS 知识库加载成功！")
except Exception as e:
    print(f"❌ FAISS 知识库加载失败，请先运行 build.py。\n详细信息: {e}")
    exit()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PromptTemplate.from_template(
        "根据以下背景知识回答问题。\n\n背景知识:\n{context}\n\n问题: {question}\n\n答案:"
    )
    | llm
    | StrOutputParser()
)

# ---------- PDF 知识库 ----------
pdf_vectorstore = None
pdf_path = "mathandphyeqs.pdf"

if os.path.exists(pdf_path):
    print("正在加载 PDF 知识库...")
    try:
        pdf_loader = PyPDFLoader(pdf_path)
        raw_pages = pdf_loader.load()
        pdf_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        pdf_docs = pdf_splitter.split_documents(raw_pages)
        pdf_vectorstore = FAISS.from_documents(pdf_docs, embeddings)  # 复用同一 embeddings
        print(f"✅ PDF 知识库加载成功！共 {len(pdf_docs)} 个片段。")
    except Exception as e:
        print(f"⚠️  PDF 加载失败，将跳过 PDF 工具。\n详细信息: {e}")
else:
    print("⚠️  未找到 PDF 文件，将跳过 PDF 工具。")

# =======================================================
# 第三步：定义工具
# =======================================================

# 工具1：原有 FAISS 知识库检索
def retrieve_knowledge(query: str) -> str:
    return rag_chain.invoke(query)

# 工具2：PDF 原文检索
def pdf_quick_search(query: str) -> str:
    if pdf_vectorstore is None:
        return "PDF 知识库未加载，无法使用此工具。"
    results = pdf_vectorstore.similarity_search(query, k=3)
    return "\n---\n".join([res.page_content for res in results])

# 工具3：SymPy 数理计算引擎（替代原有 numexpr Calculator）
def physics_math_solver(query: str) -> str:
    """
    执行 Python 数理计算代码。
    支持 sympy（符号运算：求导、积分、方程求解）和 math/numpy（数值计算）。
    """
    safe_globals = {
        "sympy": __import__("sympy"),
        "math":  __import__("math"),
        "np":    __import__("numpy"),
        # 常用 sympy 函数直接暴露，方便模型调用
        "symbols":   __import__("sympy").symbols,
        "solve":     __import__("sympy").solve,
        "diff":      __import__("sympy").diff,
        "integrate": __import__("sympy").integrate,
        "simplify":  __import__("sympy").simplify,
        "pi":        __import__("sympy").pi,
        "sqrt":      __import__("sympy").sqrt,
        "print":     print,
    }

    old_stdout = sys.stdout
    sys.stdout = redirected = StringIO()
    try:
        exec(query, safe_globals)
        sys.stdout = old_stdout
        output = redirected.getvalue().strip()
        return output if output else "代码执行成功，但没有输出。请在代码中使用 print() 输出结果。"
    except Exception as e:
        sys.stdout = old_stdout
        return f"代码执行出错: {str(e)}"

# 组装工具列表（根据 PDF 是否加载动态增减）
tools = [
    Tool(
        name="Knowledge_Base",
        func=retrieve_knowledge,
        description=(
            "查询预建的物理定律、公式、概念知识库。"
            "输入是具体的中文问题，例如：牛顿第二定律是什么？"
        )
    ),
    Tool(
        name="Math_Solver",
        func=physics_math_solver,
        description=(
            "数理计算引擎，支持符号运算和数值计算。"
            "输入必须是合法的 Python 代码，使用 print() 输出结果。"
            "可直接使用 sympy、math、np、symbols、solve、diff、integrate 等。"
            "示例：print(solve('x**2 - 4', symbols('x'))) 或 print(math.sqrt(9))"
        )
    ),
]

if pdf_vectorstore is not None:
    tools.insert(1, Tool(
        name="PDF_Reference",
        func=pdf_quick_search,
        description=(
            "直接搜索《数理公式》PDF 文档中的原始文本。"
            "当需要查找特定物理公式或数学定义的原文时使用。"
            "输入是关键词或问题描述。"
        )
    ))

# =======================================================
# 第四步：容错解析器
# =======================================================
FINAL_ANSWER_TRIGGERS = [
    "Final Answer:",
    "最终答案",
    "**最终答案",
    "final answer:",
    "最终回答",
    "答案：",
    "答案:",
]

class ForgivingOutputParser(ReActSingleInputOutputParser):
    def parse(self, text: str):
        for trigger in FINAL_ANSWER_TRIGGERS:
            if trigger in text:
                answer = text.split(trigger, 1)[-1].strip().strip("*").strip()
                return AgentFinish(return_values={"output": answer}, log=text)
        # 兜底：内容完整但无 Action，直接返回
        if len(text.strip()) > 50 and "Action:" not in text:
            return AgentFinish(return_values={"output": text.strip()}, log=text)
        return super().parse(text)

# =======================================================
# 第五步：构建 ReAct Agent
# =======================================================
prompt_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format STRICTLY:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

IMPORTANT RULES:
- For Math_Solver, Action Input must be valid Python code that uses print() to output results.
- Always end your response with "Final Answer:" followed by your complete answer.
- Do NOT use markdown headers or **bold** before "Final Answer:".
- If you already know the answer, go directly to "Final Answer:".

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate.from_template(prompt_template)
agent = create_react_agent(llm, tools, prompt, output_parser=ForgivingOutputParser())

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors="请严格按照格式，以 'Final Answer:' 结尾。",
    max_iterations=10
)

# =======================================================
# 第六步：交互式运行
# =======================================================
if __name__ == "__main__":
    tool_names = [t.name for t in tools]
    print("\n" + "="*50)
    print(f">>> 数理全能王智能体已就绪！可用工具: {tool_names}")
    print(">>> 示例问题：")
    print("    1. 根据牛顿第二定律，质量5kg加速度2m/s^2，力是多少？")
    print("    2. 求 x^2 - 5x + 6 = 0 的解")
    print("    3. 对 sin(x) 求导")
    print("    (输入 'exit' 退出)")
    print("="*50 + "\n")

    while True:
        try:
            user_input = input("用户提问: ").strip()
            if user_input.lower() in ["exit", "quit", "q", "退出"]:
                print("再见！")
                break
            if not user_input:
                continue
            print("-" * 30)
            result = agent_executor.invoke({"input": user_input})
            print("-" * 30)
            print(f"最终答案: {result['output']}")
            print("=" * 50 + "\n")
        except KeyboardInterrupt:
            print("\n再见！")
            break
        except Exception as e:
            print(f"发生错误: {e}")
            print("请尝试换个问法。\n")